# AgroPredict — Model Training v4 (final: from scratch to deployment)

**Architecture decision (made here, explained, not left as an open question):**

- **Single model, all 140 crops** — not a two-stage cluster-then-crop system. Simpler to build,
  train, and deploy, and the shortlist framing below gets most of the same benefit without the
  extra complexity.
- **Model A features only** (agronomic + `soil_type`/`season` + engineered ratios). Geography
  (`state`/`lat`/`long`) added only +0.016 top-3 accuracy in the v3 comparison — not worth the
  extra deployment dependency on a location lookup succeeding.
- **XGBoost** — won every comparison run so far (v2 and v3, Model A and Model B, both metrics),
  so it's the model this notebook tunes and ships.
- **Delivered as a top-5 shortlist with probabilities, not a single answer** — this is the direct
  response to what the data analysis found: 127 of 140 crops have at least one agro-climatically
  near-identical "twin" crop (some clusters as large as 56 crops), because `crop_profiles.csv`
  assigns many crops nearly the same temperature/rainfall/pH/season/soil ranges. A single exact-crop
  prediction can't honestly reflect that; a ranked shortlist can.

**Evaluation uses four metrics together**, each answering a different question:
- `macro_f1` — strict, single-guess correctness (kept for comparability with earlier runs)
- `top3_acc` / `top5_acc` — was the true crop in the model's top 3 / top 5 guesses
- `top5_cluster_acc` — was the true crop's *agro-climatic cluster* represented in the top 5 guesses
  (the most realistic measure of "did this shortlist actually make sense")

Every expensive cell is checkpointed to a `.jsonl` file on Drive and safely resumable across
Colab disconnects, using single-level parallelism (model parallelizes internally, the CV loop
itself runs sequentially) to avoid the RAM crashes from earlier attempts.


## 1. Colab Setup

In [ ]:
import os

# EDIT THIS to the folder in your Drive that contains the CSVs
DATA_DIR = "/content/drive/MyDrive/Dataset For colab/AgroPredict V2"

expected_files = [
    "indian_crop_training_dataset.csv",
    "crop_profiles.csv",
    "data_dictionary.csv",
    "real_indian_crop_validation_dataset.csv",
]
missing = [f for f in expected_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print("Could not find these files in DATA_DIR -- fix the path above:")
    for f in missing:
        print(" -", f)
    print()
    print("Contents of DATA_DIR:")
    print(os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "DATA_DIR does not exist")
else:
    print("All expected files found in", DATA_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost lightgbm shap joblib


## 2. Imports, Data Load, Feature Engineering

In [ ]:
import gc
import json
import itertools
import warnings
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, ParameterSampler
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, top_k_accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", message="X does not have valid feature names")

df = pd.read_csv(os.path.join(DATA_DIR, "indian_crop_training_dataset.csv"))
crop_profiles = pd.read_csv(os.path.join(DATA_DIR, "crop_profiles.csv"))

RAW_NUMERIC_FEATURES = [
    "nitrogen_N_kg_ha", "phosphorus_P_kg_ha", "potassium_K_kg_ha",
    "temperature_C", "humidity_percent", "rainfall_mm",
    "soil_pH", "soil_moisture_percent",
]
CATEGORICAL_FEATURES = ["soil_type", "season"]
TARGET = "crop"

# --- Feature engineering: domain-informed ratios/indices ---
EPS = 1e-3
df["npk_sum"] = df["nitrogen_N_kg_ha"] + df["phosphorus_P_kg_ha"] + df["potassium_K_kg_ha"]
df["n_p_ratio"] = df["nitrogen_N_kg_ha"] / (df["phosphorus_P_kg_ha"] + EPS)
df["n_k_ratio"] = df["nitrogen_N_kg_ha"] / (df["potassium_K_kg_ha"] + EPS)
df["p_k_ratio"] = df["phosphorus_P_kg_ha"] / (df["potassium_K_kg_ha"] + EPS)
df["aridity_index"] = df["rainfall_mm"] / (df["temperature_C"] + EPS)

ENGINEERED_FEATURES = ["npk_sum", "n_p_ratio", "n_k_ratio", "p_k_ratio", "aridity_index"]
NUMERIC_FEATURES = RAW_NUMERIC_FEATURES + ENGINEERED_FEATURES
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

for col in NUMERIC_FEATURES:
    df[col] = df[col].astype("float32")
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")

print(f"Loaded {df.shape[0]:,} rows, {df[TARGET].nunique()} crop classes")
print(f"Features ({len(FEATURES)}): {FEATURES}")


## 3. Data-Driven Crop Clusters — the real reason exact-crop accuracy plateaus

For every pair of crops, compare their `crop_profiles.csv` temperature/rainfall/pH ranges plus
season and soil-type overlap. Crops that are >85% overlapping on all three ranges AND share a
season AND share a soil type are agro-climatically near-identical by the dataset's own design --
no feature set derived from this data can reliably separate them. Group these into clusters via
union-find; this becomes the basis for the `top5_cluster_acc` metric used throughout.


In [ ]:
def overlap_fraction(a_min, a_max, b_min, b_max):
    inter = max(0, min(a_max, b_max) - max(a_min, b_min))
    union = max(a_max, b_max) - min(a_min, b_min)
    return inter / union if union > 0 else 0

parent = {c: c for c in crop_profiles["crop"]}
def find(x):
    while parent[x] != x:
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

profile_records = crop_profiles.to_dict("records")
for a, b in itertools.combinations(profile_records, 2):
    t_ov = overlap_fraction(a["min_temperature_C"], a["max_temperature_C"], b["min_temperature_C"], b["max_temperature_C"])
    r_ov = overlap_fraction(a["min_rainfall_mm"], a["max_rainfall_mm"], b["min_rainfall_mm"], b["max_rainfall_mm"])
    p_ov = overlap_fraction(a["min_pH"], a["max_pH"], b["min_pH"], b["max_pH"])
    same_season = bool(set(str(a["season"]).split("|")) & set(str(b["season"]).split("|")))
    same_soil = bool(set(str(a["soil_types"]).split("|")) & set(str(b["soil_types"]).split("|")))
    combined = np.mean([t_ov, r_ov, p_ov])
    if combined > 0.85 and same_season and same_soil:
        union(a["crop"], b["crop"])

crop_to_cluster = {c: find(c) for c in parent}
cluster_sizes = pd.Series(crop_to_cluster).value_counts()

print(f"{cluster_sizes.shape[0]} clusters found across {len(crop_to_cluster)} crops")
print(f"Largest cluster: {cluster_sizes.max()} crops")
print(f"Crops with no near-twin (cluster size 1): {(cluster_sizes[cluster_sizes.index.map(lambda k: (pd.Series(crop_to_cluster)==k).sum())] == 1).sum() if False else (pd.Series(crop_to_cluster).map(cluster_sizes) == 1).sum()}")
print()
print("5 largest clusters:")
for cluster_id, size in cluster_sizes.head(5).items():
    members = [c for c, cl in crop_to_cluster.items() if cl == cluster_id]
    print(f"  ({size} crops) {members}")

with open(os.path.join(DATA_DIR, "crop_clusters.json"), "w") as f:
    json.dump(crop_to_cluster, f, indent=2)


## 4. Train/Test Split (stratified, held out until Section 9)

In [ ]:
le = LabelEncoder()
X = df[FEATURES].copy()
y = le.fit_transform(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")

# Lookup array: cluster_of_label[encoded_label] -> cluster id string.
# Built in the SAME order as le.classes_ so it aligns with y's integer encoding everywhere.
cluster_of_label = np.array([crop_to_cluster[c] for c in le.classes_])
N_CLASSES = len(le.classes_)
ALL_LABELS = np.arange(N_CLASSES)


## 5. Preprocessing Pipelines

In [ ]:
def make_preprocessor(scale_numeric):
    num_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), CATEGORICAL_FEATURES),
        ("num", num_step, NUMERIC_FEATURES),
    ])

preprocessor_tree = make_preprocessor(scale_numeric=False)
preprocessor_scaled = make_preprocessor(scale_numeric=True)
print("Preprocessors ready.")


## 6. Evaluation Metrics — macro-F1, top-3, top-5, and top-5 cluster accuracy

In [ ]:
def top_k_scorer(estimator, X, y, k):
    proba = estimator.predict_proba(X)
    return top_k_accuracy_score(y, proba, k=k, labels=ALL_LABELS)

def top5_cluster_scorer(estimator, X, y):
    proba = estimator.predict_proba(X)
    top5_idx = np.argsort(proba, axis=1)[:, -5:]
    y_arr = np.asarray(y)
    true_cluster = cluster_of_label[y_arr]
    pred_clusters = cluster_of_label[top5_idx]
    hits = [true_cluster[i] in pred_clusters[i] for i in range(len(y_arr))]
    return float(np.mean(hits))

SCORING = {
    "macro_f1": "f1_macro",
    "top3_acc": lambda est, X, y: top_k_scorer(est, X, y, k=3),
    "top5_acc": lambda est, X, y: top_k_scorer(est, X, y, k=5),
    "top5_cluster_acc": top5_cluster_scorer,
}


## 7. Baseline Comparison (checkpointed, resumable)

Checkpoint filenames match v3's Model A run -- if you already ran that comparison in this same
`DATA_DIR`, those 7 models will show as already scored (for the old metrics). Since this notebook
adds two new metrics (`top5_acc`, `top5_cluster_acc`), it uses fresh checkpoint files so all four
metrics are computed consistently for every model.


In [ ]:
def run_baseline_comparison(model_dict, preprocessor, X, y, checkpoint_path, cv, svc_subsample=None):
    completed = {}
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            for line in f:
                rec = json.loads(line)
                completed[rec["name"]] = rec
        print(f"Resuming: {len(completed)} model(s) already scored: {list(completed.keys())}")

    for name, model in model_dict.items():
        if name in completed:
            rec = completed[name]
            print(f"{name:<20} macro-F1: {rec['macro_f1']:.4f} | top3: {rec['top3_acc']:.4f} | top5: {rec['top5_acc']:.4f} | top5-cluster: {rec['top5_cluster_acc']:.4f}  [skipped]")
            continue

        this_X, this_y = X, y
        note = ""
        if name == "SVC" and svc_subsample is not None:
            this_X, _, this_y, _ = train_test_split(X, y, train_size=svc_subsample, stratify=y, random_state=42)
            note = f"estimated from a {svc_subsample:,}-row subsample"

        pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
        scores = cross_validate(pipe, this_X, this_y, cv=cv, scoring=SCORING, n_jobs=1, pre_dispatch="1*n_jobs")

        rec = {
            "name": name,
            "macro_f1": float(np.mean(scores["test_macro_f1"])),
            "top3_acc": float(np.mean(scores["test_top3_acc"])),
            "top5_acc": float(np.mean(scores["test_top5_acc"])),
            "top5_cluster_acc": float(np.mean(scores["test_top5_cluster_acc"])),
            "note": note,
        }
        with open(checkpoint_path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        completed[name] = rec
        print(f"{name:<20} macro-F1: {rec['macro_f1']:.4f} | top3: {rec['top3_acc']:.4f} | top5: {rec['top5_acc']:.4f} | top5-cluster: {rec['top5_cluster_acc']:.4f}{'  [' + note + ']' if note else ''}")
        del pipe, scores
        gc.collect()

    return completed


cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

no_scale_models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1),
}
scale_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "SVC": SVC(random_state=42, probability=True),
}

CHECKPOINT_TREE = os.path.join(DATA_DIR, "baseline_v4_tree_checkpoint.jsonl")
CHECKPOINT_SCALED = os.path.join(DATA_DIR, "baseline_v4_scaled_checkpoint.jsonl")

results = {}
results.update(run_baseline_comparison(no_scale_models, preprocessor_tree, X_train, y_train, CHECKPOINT_TREE, cv5))
results.update(run_baseline_comparison(scale_models, preprocessor_scaled, X_train, y_train, CHECKPOINT_SCALED, cv5, svc_subsample=15000))


## 8. Results and Winner Selection

In [ ]:
results_df = pd.DataFrame(results).T[["macro_f1", "top3_acc", "top5_acc", "top5_cluster_acc", "note"]].sort_values("top5_cluster_acc", ascending=False)
print(results_df)
results_df.to_csv(os.path.join(DATA_DIR, "baseline_v4_results.csv"))

WINNER = results_df.index[0]
print(f"\nWinner (by top5_cluster_acc): {WINNER}")


## 9. Hyperparameter Tuning (checkpointed, resumable)

In [ ]:
PARAM_GRIDS = {
    "RandomForest": {
        "model__n_estimators": [200, 400, 600, 800],
        "model__max_depth": [10, 20, 30, 50],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
    },
    "XGBoost": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [4, 6, 8, 10],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    },
    "LightGBM": {
        "model__n_estimators": [200, 400, 600],
        "model__num_leaves": [31, 63, 127],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    },
    "DecisionTree": {
        "model__max_depth": [None, 10, 20, 30, 50],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
}

all_models = {**no_scale_models, **scale_models}
scaled_needed = WINNER in ("LogisticRegression", "KNN", "SVC")
winner_preprocessor = preprocessor_scaled if scaled_needed else preprocessor_tree
base_model = clone(all_models[WINNER])

CHECKPOINT_PATH = os.path.join(DATA_DIR, f"search_v4_{WINNER}_checkpoint.jsonl")
N_ITER = 10
SEARCH_CV_FOLDS = 3
search_cv = StratifiedKFold(n_splits=SEARCH_CV_FOLDS, shuffle=True, random_state=42)
candidates = list(ParameterSampler(PARAM_GRIDS[WINNER], n_iter=N_ITER, random_state=42))

completed = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            completed[json.dumps(rec["params"], sort_keys=True)] = rec
    print(f"Resuming: {len(completed)} candidates already completed.")

for i, params in enumerate(candidates):
    key = json.dumps(params, sort_keys=True)
    if key in completed:
        rec = completed[key]
        print(f"[{i+1}/{N_ITER}] already done: top5-cluster {rec['top5_cluster_acc']:.4f} | {params}")
        continue

    pipe = Pipeline([("preprocess", winner_preprocessor), ("model", clone(base_model))])
    pipe.set_params(**params)
    scores = cross_validate(pipe, X_train, y_train, cv=search_cv, scoring=SCORING, n_jobs=1)

    rec = {
        "params": params,
        "macro_f1": float(np.mean(scores["test_macro_f1"])),
        "top3_acc": float(np.mean(scores["test_top3_acc"])),
        "top5_acc": float(np.mean(scores["test_top5_acc"])),
        "top5_cluster_acc": float(np.mean(scores["test_top5_cluster_acc"])),
    }
    with open(CHECKPOINT_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n")
    completed[key] = rec
    print(f"[{i+1}/{N_ITER}] top5-cluster: {rec['top5_cluster_acc']:.4f} | macro-F1: {rec['macro_f1']:.4f} | {params}")
    del pipe, scores
    gc.collect()

best = max(completed.values(), key=lambda r: r["top5_cluster_acc"])
print(f"\nBest candidate: top5-cluster {best['top5_cluster_acc']:.4f} | {best['params']}")

final_model = Pipeline([("preprocess", winner_preprocessor), ("model", clone(base_model))])
final_model.set_params(**best["params"])
final_model.fit(X_train, y_train)
print("final_model fit on full training set.")


## 10. Final Evaluation — Held-out Test Set

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

test_macro_f1 = f1_score(y_test, y_pred, average="macro")
test_top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=ALL_LABELS)
test_top5 = top_k_accuracy_score(y_test, y_proba, k=5, labels=ALL_LABELS)

top5_idx_test = np.argsort(y_proba, axis=1)[:, -5:]
true_cluster_test = cluster_of_label[y_test]
pred_clusters_test = cluster_of_label[top5_idx_test]
test_top5_cluster = np.mean([true_cluster_test[i] in pred_clusters_test[i] for i in range(len(y_test))])

print(f"Held-out macro-F1:          {test_macro_f1:.4f}")
print(f"Held-out top-3 accuracy:    {test_top3:.4f}")
print(f"Held-out top-5 accuracy:    {test_top5:.4f}")
print(f"Held-out top-5 cluster acc: {test_top5_cluster:.4f}   <- the realistic 'useful shortlist' number")

report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
report_df = pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
print("\nWeakest-performing crops:")
print(report_df.sort_values("f1-score").head(10))

# --- Concrete demo: what the shortlist actually looks like for real test rows ---
print("\n--- Sample shortlist outputs ---")
sample_idx = np.random.RandomState(42).choice(len(X_test), size=5, replace=False)
for idx in sample_idx:
    true_crop = le.classes_[y_test[idx]]
    proba_row = y_proba[idx]
    top5 = np.argsort(proba_row)[-5:][::-1]
    shortlist = [(le.classes_[i], round(float(proba_row[i]), 3)) for i in top5]
    hit = "HIT" if y_test[idx] in top5 else ("CLUSTER HIT" if crop_to_cluster[true_crop] in [crop_to_cluster[c] for c, _ in shortlist] else "MISS")
    print(f"True: {true_crop:<20} Shortlist: {shortlist}  [{hit}]")


## 11. Explainability — SHAP

In [ ]:
import shap

SHAP_SAMPLE_SIZE = 500
X_shap_sample = X_test.sample(n=min(SHAP_SAMPLE_SIZE, len(X_test)), random_state=42)
X_shap_transformed = final_model.named_steps["preprocess"].transform(X_shap_sample)
feature_names_out = final_model.named_steps["preprocess"].get_feature_names_out()

fitted_model = final_model.named_steps["model"]
tree_based = isinstance(fitted_model, (RandomForestClassifier, DecisionTreeClassifier, XGBClassifier, LGBMClassifier))

if tree_based:
    explainer = shap.TreeExplainer(fitted_model)
    shap_values = explainer.shap_values(X_shap_transformed)
else:
    explainer = shap.Explainer(fitted_model.predict_proba, X_shap_transformed)
    shap_values = explainer(X_shap_transformed).values

if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.Series(mean_abs_shap, index=feature_names_out).sort_values(ascending=False)
print("Mean |SHAP value| by feature (overall importance across all 140 classes):")
print(shap_importance.head(20))


## 12. Save Artifacts for FastAPI Deployment

In [ ]:
joblib.dump(final_model, os.path.join(DATA_DIR, "model_final_pipeline.joblib"))
joblib.dump(le, os.path.join(DATA_DIR, "target_label_encoder.joblib"))

deploy_metadata = {
    "features": FEATURES,
    "categorical_options": {col: sorted(df[col].astype(str).unique().tolist()) for col in CATEGORICAL_FEATURES},
    "n_classes": int(N_CLASSES),
    "crop_to_cluster": crop_to_cluster,
    "test_macro_f1": float(test_macro_f1),
    "test_top3_acc": float(test_top3),
    "test_top5_acc": float(test_top5),
    "test_top5_cluster_acc": float(test_top5_cluster),
    "output_format": "top-5 shortlist with probabilities, not a single crop",
}
with open(os.path.join(DATA_DIR, "deploy_metadata.json"), "w") as f:
    json.dump(deploy_metadata, f, indent=2)

print("Saved: model_final_pipeline.joblib, target_label_encoder.joblib, deploy_metadata.json, crop_clusters.json")


## 13. Summary — for your portfolio write-up

- **Problem:** recommend suitable crops for Indian agricultural conditions from soil + climate
  data, delivered as an end-to-end API + frontend tool.
- **Initial approach:** single-label 140-way classification. Every model family tried (KNN through
  XGBoost) plateaued at a similar low macro-F1, ruling out a pipeline bug.
- **Root-cause analysis:** quantified pairwise overlap between all 140 crops' documented
  climate/soil requirement ranges. Found 127/140 crops (91%) have at least one near-identical
  "twin," clustering into 21 groups (one as large as 56 crops) -- meaning a large share of crops
  are not reliably separable from these features alone, by design of the source data.
- **Redesign:** reframed the deliverable from "predict the one correct crop" to "return a ranked
  top-5 shortlist with probabilities," and added a cluster-aware evaluation metric
  (`top5_cluster_acc`) that measures what actually matters for the use case: did the shortlist
  land in the right agro-climatic neighborhood.
- **Result:** [fill in your actual `test_top5_cluster_acc` and `test_top5_acc` numbers here once
  this notebook finishes running].
- **Engineering practices applied:** stratified train/test split held out until final evaluation,
  domain-informed feature engineering, checkpointed/resumable training across 7 model families,
  single-level parallelism to avoid RAM exhaustion, SHAP-based explainability, and serialized
  artifacts + metadata (including the cluster map) ready for FastAPI deployment.
